# MF Analytics Platform — Exploratory Data Analysis
## Bluestock Fintech Capstone · EDA Notebook (15+ Charts)
**Date:** June 2025

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14

DB_PATH = Path('..') / 'mf_analytics.db'
conn = sqlite3.connect(DB_PATH)
print('Connected to:', DB_PATH)

## 1. Fund Master Overview

In [ ]:
funds = pd.read_sql('SELECT * FROM dim_fund', conn)
print(f'Total schemes: {len(funds)}')
print(f'AMCs: {funds["amc_name"].nunique()}')
print('\nCategory breakdown:')
print(funds['category'].value_counts())
print('\nSub-category breakdown:')
print(funds['sub_category'].value_counts())

In [ ]:
# Chart 1: Scheme count by category
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
funds['category'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Chart 1: Scheme Count by Category')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Number of Schemes')
axes[0].tick_params(axis='x', rotation=0)

# Chart 2: Expense ratio distribution by category
funds.boxplot(column='expense_ratio', by='category', ax=axes[1])
axes[1].set_title('Chart 2: Expense Ratio Distribution by Category')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Expense Ratio (%)')
plt.suptitle('')
plt.tight_layout()
plt.savefig('../data/processed/chart_01_02_fund_master.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. AUM Analysis

In [ ]:
aum = pd.read_sql('SELECT * FROM fact_aum', conn)
aum['report_month'] = pd.to_datetime(aum['report_month'])

# Chart 3: AUM by AMC — latest month
latest = aum[aum['report_month'] == aum['report_month'].max()]
latest_grp = latest.groupby('amc_name')['aum_crore'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.Blues_r(np.linspace(0.3, 0.9, len(latest_grp)))
bars = ax.bar(latest_grp.index, latest_grp.values / 1000, color=colors, edgecolor='white')
ax.set_title('Chart 3: AUM by AMC — Latest Month (₹ Thousand Crore)')
ax.set_ylabel('AUM (₹ Thousand Crore)')
ax.set_xlabel('AMC')
for bar, val in zip(bars, latest_grp.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'₹{val/1000:.0f}K Cr',
            ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../data/processed/chart_03_aum_by_amc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 4: Industry AUM trend 2020–2025
industry_aum = aum.groupby('report_month')['aum_crore'].sum().reset_index()

fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(industry_aum['report_month'], industry_aum['aum_crore']/100000,
                alpha=0.3, color='steelblue')
ax.plot(industry_aum['report_month'], industry_aum['aum_crore']/100000,
        color='steelblue', linewidth=2)
ax.set_title('Chart 4: Industry AUM Trend 2020–2025 (₹ Lakh Crore)')
ax.set_ylabel('AUM (₹ Lakh Crore)')
ax.set_xlabel('Month')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('₹%.0f L Cr'))
plt.tight_layout()
plt.savefig('../data/processed/chart_04_aum_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 5: AMC market share — stacked area
aum_pivot = aum.pivot_table(index='report_month', columns='amc_name', values='aum_crore', aggfunc='sum')
aum_pct = aum_pivot.div(aum_pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(13, 6))
aum_pct.plot(kind='area', stacked=True, ax=ax, alpha=0.8)
ax.set_title('Chart 5: AMC Market Share (%) — Stacked Area 2020–2025')
ax.set_ylabel('Market Share (%)')
ax.set_xlabel('Month')
ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig('../data/processed/chart_05_market_share.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. SIP Analysis

In [ ]:
sip = pd.read_sql('SELECT * FROM fact_sip', conn)
sip['report_month'] = pd.to_datetime(sip['report_month'])
industry_sip = sip.groupby('report_month')['sip_amount_crore'].sum().reset_index()

# Chart 6: Monthly SIP inflow trend
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(industry_sip['report_month'], industry_sip['sip_amount_crore'],
       color='coral', alpha=0.85, width=20)
# Add 3-month rolling average
rolling = industry_sip['sip_amount_crore'].rolling(3).mean()
ax.plot(industry_sip['report_month'], rolling, color='darkred', linewidth=2, label='3M Rolling Avg')
ax.set_title('Chart 6: Monthly Industry SIP Inflow (₹ Crore) — 2020–2025')
ax.set_ylabel('SIP Inflow (₹ Crore)')
ax.set_xlabel('Month')
ax.legend()
ax.axhline(y=31002, color='green', linestyle='--', alpha=0.7, label='Dec 25 ₹31,002 Cr')
plt.tight_layout()
plt.savefig('../data/processed/chart_06_sip_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 7: SIP inflow by AMC — latest month
sip_latest = sip[sip['report_month'] == sip['report_month'].max()]
sip_amc = sip_latest.groupby('amc_name')['sip_amount_crore'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
sip_amc.plot(kind='barh', ax=ax, color='coral', edgecolor='white')
ax.set_title('Chart 7: SIP Inflow by AMC — Latest Month (₹ Crore)')
ax.set_xlabel('SIP Inflow (₹ Crore)')
plt.tight_layout()
plt.savefig('../data/processed/chart_07_sip_by_amc.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. NAV Distribution Analysis

In [ ]:
nav = pd.read_sql('''
    SELECT n.nav_value, n.day_change_pct, f.category, f.sub_category, f.scheme_name
    FROM fact_nav n JOIN dim_fund f ON n.fund_id = f.fund_id
    WHERE n.nav_date = (SELECT MAX(nav_date) FROM fact_nav)
''', conn)

# Chart 8: NAV distribution by category (violin plot)
fig, ax = plt.subplots(figsize=(12, 5))
nav_plot = nav[nav['nav_value'] < nav['nav_value'].quantile(0.95)]
sns.violinplot(data=nav_plot, x='category', y='nav_value', ax=ax, palette='muted')
ax.set_title('Chart 8: NAV Distribution by Category (Latest)')
ax.set_ylabel('NAV (₹)')
ax.set_xlabel('Category')
plt.tight_layout()
plt.savefig('../data/processed/chart_08_nav_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 9: Daily return distribution (histogram + KDE)
all_returns = pd.read_sql('SELECT day_change_pct FROM fact_nav WHERE day_change_pct IS NOT NULL', conn)
returns_clean = all_returns['day_change_pct'].dropna()
returns_clean = returns_clean[returns_clean.between(-10, 10)]  # remove extreme outliers

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(returns_clean, bins=100, density=True, alpha=0.6, color='steelblue', edgecolor='none')
returns_clean.plot(kind='kde', ax=ax, color='darkblue', linewidth=2)
ax.axvline(x=0, color='red', linestyle='--', alpha=0.7)
ax.set_title('Chart 9: Daily NAV Return Distribution (All Funds)')
ax.set_xlabel('Daily Return (%)')
ax.set_ylabel('Density')
plt.tight_layout()
plt.savefig('../data/processed/chart_09_return_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Investor Demographics

In [ ]:
txn = pd.read_sql('SELECT * FROM fact_transactions', conn)
txn['txn_date'] = pd.to_datetime(txn['txn_date'])

# Chart 10: Investor count by state
state_inv = txn[txn['txn_type'].isin(['BUY','SIP'])].groupby('state')['investor_id'].nunique().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#d73027' if s in ['Maharashtra','Karnataka','Tamil Nadu','Delhi','Gujarat'] else '#4575b4'
          for s in state_inv.index]
state_inv.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.set_title('Chart 10: Unique Investors by State (Top 5 highlighted in red)')
ax.set_ylabel('Unique Investors')
ax.set_xlabel('State')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../data/processed/chart_10_state_investors.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 11: Age distribution histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
txn['investor_age'].hist(bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Chart 11a: Investor Age Distribution')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].axvline(x=30, color='red', linestyle='--', label='Age 30')
axes[0].legend()

# SIP vs Lumpsum pie
sip_lump = txn[txn['txn_type'].isin(['BUY','SIP'])].groupby(
    txn['txn_type'].apply(lambda x: 'SIP' if x == 'SIP' else 'Lumpsum')
)['txn_amount'].sum()
axes[1].pie(sip_lump.values, labels=sip_lump.index, autopct='%1.1f%%',
            colors=['coral','steelblue'], startangle=90)
axes[1].set_title('Chart 11b: SIP vs Lumpsum — Total Investment Split')
plt.tight_layout()
plt.savefig('../data/processed/chart_11_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 12: Channel mix
channel = txn.groupby('channel')['txn_amount'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
channel.plot(kind='bar', ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('Chart 12a: Transaction Volume by Channel (₹)')
axes[0].set_xlabel('Channel')
axes[0].set_ylabel('Total Amount (₹)')

# City tier
tier = txn.groupby('city_tier').agg(investors=('investor_id','nunique'), amount=('txn_amount','sum'))
tier['amount'].plot(kind='bar', ax=axes[1], color=['#2166ac','#f4a582','#d6604d'], edgecolor='white')
axes[1].set_title('Chart 12b: Investment by City Tier (₹)')
axes[1].set_xlabel('City Tier')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('../data/processed/chart_12_channel_tier.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Time Series & Correlation Analysis

In [ ]:
# Chart 13: NAV trend for top 5 schemes (by latest NAV)
nav_ts = pd.read_sql('''
    SELECT n.nav_date, n.nav_value, f.scheme_name, f.sub_category
    FROM fact_nav n JOIN dim_fund f ON n.fund_id = f.fund_id
    WHERE f.sub_category IN ('Large Cap','Mid Cap','Small Cap')
    AND n.nav_date >= '2022-01-01'
''', conn)
nav_ts['nav_date'] = pd.to_datetime(nav_ts['nav_date'])

# Pick one representative scheme per sub-category
top_schemes = nav_ts.groupby('scheme_name')['nav_value'].last().nlargest(6).index.tolist()
nav_ts_top = nav_ts[nav_ts['scheme_name'].isin(top_schemes)]

fig, ax = plt.subplots(figsize=(13, 6))
for name, grp in nav_ts_top.groupby('scheme_name'):
    ax.plot(grp['nav_date'], grp['nav_value'], label=name[:35], linewidth=1.5)
ax.set_title('Chart 13: NAV Trend — Top Schemes (Jan 2022–Jun 2025)')
ax.set_ylabel('NAV (₹)')
ax.set_xlabel('Date')
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.savefig('../data/processed/chart_13_nav_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 14: Correlation heatmap of daily returns across categories
returns_wide = pd.read_sql('''
    SELECT n.nav_date, f.sub_category, AVG(n.day_change_pct) as avg_return
    FROM fact_nav n JOIN dim_fund f ON n.fund_id = f.fund_id
    WHERE n.nav_date >= '2022-01-01' AND n.day_change_pct IS NOT NULL
    GROUP BY n.nav_date, f.sub_category
''', conn)
returns_pivot = returns_wide.pivot(index='nav_date', columns='sub_category', values='avg_return')
corr = returns_pivot.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Chart 14: Return Correlation Heatmap — Fund Sub-Categories')
plt.tight_layout()
plt.savefig('../data/processed/chart_14_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 15: Monthly transaction volume trend
txn['month'] = txn['txn_date'].dt.to_period('M')
monthly_txn = txn.groupby(['month','txn_type'])['txn_amount'].sum().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(13, 5))
monthly_txn[['BUY','SIP']].plot(kind='area', stacked=True, ax=ax,
                                  color=['steelblue','coral'], alpha=0.8)
ax.set_title('Chart 15: Monthly Transaction Volume — BUY vs SIP (₹)')
ax.set_ylabel('Transaction Amount (₹)')
ax.set_xlabel('Month')
plt.tight_layout()
plt.savefig('../data/processed/chart_15_monthly_txn.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 16: AUM vs SIP inflow — dual axis trend
fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

ax1.plot(industry_aum['report_month'], industry_aum['aum_crore']/100000,
         color='steelblue', linewidth=2, label='AUM (₹ L Cr)')
ax2.bar(industry_sip['report_month'], industry_sip['sip_amount_crore'],
        color='coral', alpha=0.5, width=20, label='SIP Inflow (₹ Cr)')

ax1.set_ylabel('AUM (₹ Lakh Crore)', color='steelblue')
ax2.set_ylabel('SIP Inflow (₹ Crore)', color='coral')
ax1.set_title('Chart 16: Industry AUM vs SIP Inflow — Dual Axis')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.savefig('../data/processed/chart_16_aum_sip_dual.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
conn.close()
print('EDA complete. All 16 charts saved to data/processed/')